<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


- Install the additional package requirements for this bonus notebook by uncommenting and running the following cell:

In [1]:
pip install -r requirements-extra.txt

Note: you may need to restart the kernel to use updated packages.


# Comparing Various Byte Pair Encoding (BPE) Implementations

<br>
&nbsp;

## Using BPE from `tiktoken`

In [4]:
!pip install tiktoken


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 995.3/995.3 kB 3.2 MB/s eta 0:00:00? eta -:--:--


In [6]:
from importlib.metadata import version

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [18]:
import tiktoken

tik_tokenizer = tiktoken.get_encoding("gpt2")

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
)

In [20]:
integers = tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [22]:
strings = tik_tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [6]:
print(tik_tokenizer.n_vocab)

50257


<br>
&nbsp;

## Using the original BPE implementation used in GPT-2

In [24]:
from bpe_openai_gpt2 import get_encoder, download_vocab

In [26]:
download_vocab()

Fetching encoder.json: 1.04Mit [00:00, 1.29Mit/s]                                                   
Fetching vocab.bpe: 457kit [00:00, 1.15Mit/s]                                                       


In [28]:
orig_tokenizer = get_encoder(model_name="gpt2_model", models_dir=".")

In [30]:
integers = orig_tokenizer.encode(text)

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 1279, 91, 437, 1659, 5239, 91, 29, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [32]:
strings = orig_tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


<br>
&nbsp;

## Using the BPE via Hugging Face transformers

In [36]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 7.9 MB/s eta 0:00:00m eta 0:00:010:01:02


In [58]:
import transformers

transformers.__version__

'4.49.0'

In [60]:
from transformers import GPT2Tokenizer

hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [62]:
hf_tokenizer(strings)["input_ids"]

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 286,
 617,
 34680,
 27271,
 13]

In [44]:
from transformers import GPT2TokenizerFast

hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

In [46]:
hf_tokenizer_fast(strings)["input_ids"]

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 286,
 617,
 34680,
 27271,
 13]

<br>
&nbsp;

## Using my own from-scratch BPE tokenizer

In [64]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, "..", "05_bpe-from-scratch", fullname + ".ipynb")
        path = os.path.normpath(path)

        # Load the notebook
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # Create a module to store the imported functions and classes
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # Go through the notebook cells and only execute function or class definitions
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # Check for function or class definitions
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "bpe-from-scratch"
    names = ["BPETokenizerSimple"]

    return import_definitions_from_notebook(fullname, names)

In [66]:
imported_module = import_from_notebook()
BPETokenizerSimple = getattr(imported_module, "BPETokenizerSimple", None)

tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=os.path.join("gpt2_model", "encoder.json"),
    bpe_merges_path=os.path.join("gpt2_model", "vocab.bpe")
)

In [68]:
integers = tokenizer_gpt2.encode(text)

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 1279, 91, 437, 1659, 5239, 91, 29, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


<br>
&nbsp;

## A quick performance benchmark

In [70]:
with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

### Original OpenAI GPT-2 tokenizer

In [72]:
%timeit orig_tokenizer.encode(raw_text)

3.96 ms ± 15.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Tiktoken OpenAI GPT-2 tokenizer

In [74]:
%timeit tik_tokenizer.encode(raw_text)

1.01 ms ± 1.06 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Hugging Face OpenAI GPT-2 tokenizer

In [80]:
%timeit hf_tokenizer(raw_text)["input_ids"]

11.2 ms ± 36.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [78]:
%timeit hf_tokenizer(raw_text, max_length=5145, truncation=True)["input_ids"]

11.1 ms ± 68.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [82]:
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


4.3 ms ± 26.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [84]:
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)["input_ids"]

4.34 ms ± 27.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### My own GPT-2 tokenizer (for educational purposes)

In [88]:
%timeit tokenizer_gpt2.encode(raw_text)

9.46 ms ± 117 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
